# Diagnóstico del estado de los datos crudos
**Proyecto 1 — CC3084 Data Science | Actividad 3 (15 pts)**

Autor: Ernesto  
Fuente: `datos/unido/establecimientos_diversificado_unido.csv` (generado por `automatizacion/unir_datos.py`)

> **Lectura estándar del proyecto:** `encoding="utf-8-sig"`, `dtype=str`, `keep_default_na=False`, `na_values=[]`  
> Los faltantes reales son **cadenas vacías `""`** más tokens `NA / NULL / - / . / SIN DATO`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Funciona tanto desde notebooks/ (Jupyter) como desde la raíz del proyecto
ROOT = Path(".").resolve()
if not (ROOT / "datos").exists():
    ROOT = ROOT.parent

UNIDO = ROOT / "datos" / "unido" / "establecimientos_diversificado_unido.csv"
INTERIM = ROOT / "datos" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

# Tokens que actúan como faltantes en este dataset
TOKENS_FALTANTE = {"NA", "NULL", "-", ".", "SIN DATO"}

df = pd.read_csv(UNIDO, encoding="utf-8-sig", dtype=str, keep_default_na=False, na_values=[])
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

## a. Registros y variables (shape)

In [ ]:
print(f"Filas   : {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print()
print("Columnas:")
for c in df.columns:
    print(f"  {c}")

## b. Tipo de dato actual vs esperado

In [ ]:
# Todo viene como texto; describimos el tipo lógico esperado
tipo_esperado = {
    "codigo"        : "texto (ceros a la izquierda → no convertir a int)",
    "distrito"      : "texto",
    "departamento"  : "categórico texto",
    "municipio"     : "categórico texto",
    "establecimiento": "texto",
    "direccion"     : "texto",
    "telefono"      : "texto (puede tener guiones/extensiones)",
    "supervisor"    : "texto",
    "director"      : "texto",
    "nivel"         : "categórico texto (esperado: solo DIVERSIFICADO)",
    "sector"        : "categórico texto",
    "area"          : "categórico texto",
    "status"        : "categórico texto",
    "modalidad"     : "categórico texto",
    "jornada"       : "categórico texto",
    "plan"          : "categórico texto",
    "departamental" : "texto",
    "archivo_origen": "texto (trazabilidad)",
    "id_registro"   : "entero (trazabilidad)",
}

tipos_df = pd.DataFrame({
    "columna"        : df.columns,
    "tipo_actual"    : df.dtypes.values,
    "tipo_esperado"  : [tipo_esperado.get(c, "texto") for c in df.columns],
})
display(tipos_df)
tipos_df.to_csv(INTERIM / "diagnostico_tipos.csv", index=False, encoding="utf-8-sig")
print("→ guardado: datos/interim/diagnostico_tipos.csv")

## c. Faltantes por variable (vacíos `""` + tokens faltantes)

In [ ]:
def contar_faltantes(series: pd.Series) -> int:
    vacios   = (series.str.strip() == "").sum()
    por_token = series.str.strip().isin(TOKENS_FALTANTE).sum()
    return int(vacios + por_token)

total = len(df)
faltantes = pd.DataFrame({
    "columna"   : df.columns,
    "faltantes_n": [contar_faltantes(df[c]) for c in df.columns],
})
faltantes["faltantes_%"] = (faltantes["faltantes_n"] / total * 100).round(2)
faltantes = faltantes.sort_values("faltantes_%", ascending=False)
display(faltantes)
faltantes.to_csv(INTERIM / "diagnostico_faltantes.csv", index=False, encoding="utf-8-sig")
print("→ guardado: datos/interim/diagnostico_faltantes.csv")

## d. Valores únicos por variable

In [ ]:
unicos = pd.DataFrame({
    "columna" : df.columns,
    "n_unicos": [df[c].nunique() for c in df.columns],
}).sort_values("n_unicos")
display(unicos)

# value_counts para categóricas (baja cardinalidad)
CATEGORICAS = ["departamento", "nivel", "sector", "area", "status", "modalidad", "jornada", "plan"]
for col in CATEGORICAS:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().to_string())

unicos.to_csv(INTERIM / "diagnostico_unicos.csv", index=False, encoding="utf-8-sig")
print("\n→ guardado: datos/interim/diagnostico_unicos.csv")

## e. Duplicados exactos

In [ ]:
# Excluir las columnas derivadas antes de buscar duplicados exactos
cols_datos = [c for c in df.columns if c not in ("id_registro", "archivo_origen")]
n_dup = df.drop(columns=["id_registro", "archivo_origen"]).duplicated().sum()
print(f"Duplicados exactos (excluyendo id_registro + archivo_origen): {n_dup}")
assert n_dup == 0, f"¡Se encontraron {n_dup} duplicados exactos inesperados!"
print("✓ Ningún duplicado exacto — consistente con lo esperado.")

## f. Valores fuera de dominio / inconsistentes

In [ ]:
import re

# --- departamento: valores esperados (22 + la distinción 00/01 de la fuente) ---
DEPTOS_OFICIALES_FUENTE = {
    "CIUDAD CAPITAL", "GUATEMALA", "EL PROGRESO", "SACATEPÉQUEZ", "CHIMALTENANGO",
    "ESCUINTLA", "SANTA ROSA", "SOLOLÁ", "TOTONICAPÁN", "QUETZALTENANGO",
    "SUCHITEPÉQUEZ", "RETALHULEU", "SAN MARCOS", "HUEHUETENANGO", "QUICHÉ",
    "BAJA VERAPAZ", "ALTA VERAPAZ", "PETÉN", "IZABAL", "ZACAPA",
    "CHIQUIMULA", "JALAPA", "JUTIAPA"
}
fuera_depto = df[~df["departamento"].isin(DEPTOS_OFICIALES_FUENTE)]
print(f"Departamentos fuera de dominio: {len(fuera_depto)}")
if len(fuera_depto):
    display(fuera_depto[["id_registro","departamento"]].value_counts())

# --- codigo: patrón esperado ##-##-####-## ---
patron_codigo = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
fuera_codigo = df[~df["codigo"].str.match(patron_codigo, na=False)]
print(f"\nCódigos fuera de patrón ##-##-####-##: {len(fuera_codigo)}")
if len(fuera_codigo):
    display(fuera_codigo[["id_registro","codigo"]].head(10))

# --- telefono: sospechosos (letras, < 7 dígitos, cadena vacía) ---
def telefono_sospechoso(t: str) -> bool:
    t = t.strip()
    if not t or t in TOKENS_FALTANTE:
        return False  # faltante ya contado arriba
    digitos = re.sub(r"\D", "", t)
    tiene_letras = bool(re.search(r"[a-zA-Z]", t))
    return tiene_letras or len(digitos) < 7

mask_tel = df["telefono"].apply(telefono_sospechoso)
print(f"\nTeléfonos sospechosos (letras o < 7 dígitos): {mask_tel.sum()}")
if mask_tel.sum():
    display(df.loc[mask_tel, ["id_registro","telefono"]].head(15))

# --- nivel: sólo debe haber DIVERSIFICADO ---
print(f"\nValores únicos en 'nivel':")
print(df["nivel"].value_counts().to_string())

# Exportar resumen fuera de dominio
fuera_dominio = pd.concat([
    fuera_depto[["id_registro","departamento"]].assign(variable="departamento", valor=fuera_depto["departamento"]),
    fuera_codigo[["id_registro","codigo"]].assign(variable="codigo", valor=fuera_codigo["codigo"]),
    df.loc[mask_tel,["id_registro","telefono"]].assign(variable="telefono", valor=df.loc[mask_tel,"telefono"]),
], ignore_index=True)[["id_registro","variable","valor"]]
fuera_dominio.to_csv(INTERIM / "diagnostico_fuera_dominio.csv", index=False, encoding="utf-8-sig")
print("\n→ guardado: datos/interim/diagnostico_fuera_dominio.csv")

## g. Formatos inconsistentes

In [ ]:
# Espacios al inicio/fin
cols_texto = [c for c in df.columns if c not in ("id_registro","archivo_origen")]
espacios = {}
for col in cols_texto:
    n = (df[col] != df[col].str.strip()).sum()
    if n > 0:
        espacios[col] = n
print("Columnas con espacios al inicio/fin:")
print(espacios if espacios else "  Ninguna.")

# Casing: ¿algún valor en minúscula?
CATEGORICAS_CHECK = ["departamento","municipio","nivel","sector","area","status","modalidad","jornada","plan"]
print("\nRegistros con alguna minúscula en columnas categóricas:")
for col in CATEGORICAS_CHECK:
    mask = df[col].str.lower() != df[col].str.upper()
    con_minus = df.loc[mask & (df[col] != df[col].str.upper()), col].value_counts()
    if len(con_minus):
        print(f"  {col}: {len(con_minus)} valores con minúsculas")
        print(con_minus.head(5).to_string())
    else:
        print(f"  {col}: ✓ todo en MAYÚSCULAS")

# Formatos de código de distrito vs departamental
print("\nMuestra distrito:")
print(df["distrito"].value_counts().head(10).to_string())
print("\nMuestra departamental:")
print(df["departamental"].value_counts().head(10).to_string())

## h. Resumen ejecutivo — problemas y plan de limpieza

In [ ]:
resumen_filas = []
total = len(df)

for col in df.columns:
    n_falt = contar_faltantes(df[col])
    n_uniq = df[col].nunique()
    resumen_filas.append({
        "variable"    : col,
        "faltantes_n" : n_falt,
        "faltantes_%" : round(n_falt / total * 100, 2),
        "n_unicos"    : n_uniq,
        "notas"       : "",  # rellenar en el informe manual
    })

resumen = pd.DataFrame(resumen_filas)

# Agregar notas automáticas para variables con problemas detectados
notas = {
    "departamento" : "Fuente separa CIUDAD CAPITAL (00) y GUATEMALA (01); decisión de unificación pendiente.",
    "municipio"    : "Validar contra catálogo oficial (~340 municipios); posibles typos.",
    "codigo"       : f"{len(fuera_codigo)} códigos fuera de patrón ##-##-####-##.",
    "telefono"     : f"{mask_tel.sum()} registros sospechosos (letras o pocos dígitos).",
    "nivel"        : "Verificar que solo exista DIVERSIFICADO.",
    "distrito"     : "Formato variable; normalizar.",
    "departamental": "Verificar consistencia con departamento.",
}
for col, nota in notas.items():
    resumen.loc[resumen["variable"] == col, "notas"] = nota

display(resumen)
resumen.to_csv(INTERIM / "diagnostico_resumen.csv", index=False, encoding="utf-8-sig")
print("→ guardado: datos/interim/diagnostico_resumen.csv")
print()
print("=" * 60)
print("RESUMEN EJECUTIVO")
print("=" * 60)
print(f"  Registros totales : {total:,}")
print(f"  Variables         : {df.shape[1]}")
print(f"  Duplicados exactos: 0 (verificado)")
print(f"  Col. con >5% falt.: {(resumen['faltantes_%'] > 5).sum()}")
print(f"  Deptos fuera cat. : {len(fuera_depto)}")
print(f"  Códigos inválidos : {len(fuera_codigo)}")
print(f"  Teléfonos sospech.: {mask_tel.sum()}")
print()
print("Tablas guardadas en datos/interim/diagnostico_*.csv")